In [2]:
import ee
import geemap
import geopandas as gpd

ee.Authenticate()
ee.Initialize(project =  'ee-debrajkolya')

# AOI 
shp = r"D:\Google Earth Engine\GEE_MAP_Python\ee-debrajkolya\Flood & Soil moisture project\Shapefile\Ghatal.shp"

gdf = gpd.read_file(shp)
aoi = geemap.geopandas_to_ee(gdf)

# Date pre flood
pre_flood_start = '2025-04-01'
pre_flood_end = '2025-04-10'

# Date post flood
post_flood_start = '2025-09-01'
post_flood_end = '2025-09-10'

# Create function for sentinel-1 data collection
def s1(start_date, end_date):
    collection = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(aoi.geometry())
        .filterDate(start_date, end_date)
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
        .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
        .select('VV')
    )
    vv = collection.mean().clip(aoi)
    return vv

# Create function for calculate soil moisture
def calculate_soil_moisture(s1_image):
    min_max = s1_image.reduceRegion(
        reducer = ee.Reducer.minMax(),
        geometry = aoi.geometry(),
        scale = 10,
        maxPixels = 1e13
        )

    min_val = ee.Number(min_max.get('VV_min'))
    max_val = ee.Number(min_max.get('VV_max'))
    soil_moisture = (
        s1_image.subtract(min_val)
        .divide(max_val.subtract(min_val))
        .rename('Soil_Moisture')
        )
    return soil_moisture

# Call s1 function

pre_s1 = s1(pre_flood_start, pre_flood_end)
post_s1 = s1(post_flood_start, post_flood_end)

# Call calculate soil moisture function

pre_smi = calculate_soil_moisture(pre_s1)
post_smi = calculate_soil_moisture(post_s1)

difference_map = (
    post_smi.subtract(pre_smi)
    .rename("SM Difference"))

# Soil Moisture Visualization
soil_vis_pre = {
    'min': 0.4605734343081453,
    'max': 0.6836327761891605,
    'palette': ['#ffdd00', '#2bff00', '#0011ff']
}
soil_vis_post = {
    'min': 0.3526245579335332,
    'max': 0.621289865403286,
    'palette': ['#ffdd00', '#2bff00', '#0011ff']
}
# Difference Map Visualization
diff_vis = {
    'min': -0.18013488728424537,
    'max': 0.043026203105364036,
    'palette': ['#ff0000', '#ffffff', "#0033ff" ]
}

soil_legend = {
    'Low': '#ffdd00',
    'Moderate': '#2bff00',
    'High' : '#0011ff'
}

diff_legend = {
    'Decrease': '#FF0000',
    'No Change': '#FFFFFF',
    'Increase': "#0033ff"
}

Map = geemap.Map()

Map.addLayer(aoi,{}, 'Ghatal')
Map.centerObject(aoi, 11)
Map.addLayer(pre_smi, soil_vis_pre, "Pre Flood SMI")
Map.addLayer(post_smi, soil_vis_post, "Post Flood SMI")
Map.addLayer(difference_map, diff_vis, "Difference Map")

# Map.add_legend(
#     title='Soil Moisture',
#     legend_dict = soil_legend
# )

Map.add_legend(
    title='Soil Moisture Change',
    legend_dict=diff_legend
    )

Map



Map(center=[22.671034579331785, 87.6912427148179], controls=(WidgetControl(options=['position', 'transparent_b…